In [1]:
import numpy as np
from nltk.util import skipgrams
from sklearn.kernel_approximation import RBFSampler, PolynomialCountSketch
from sklearn.neighbors import NearestNeighbors

In [8]:
sent = "Insurgents killed in ongoing fighting"
for sg in list(skipgrams(sent, 4, 1)):
    print(''.join(sg), hash(''.join(sg))%10000)

Insu 2583
Insr 1285
Inur 9922
Isur 4556
nsur 387
nsug 4572
nsrg 1043
nurg 3287
surg 5038
sure 1556
suge 3726
srge 8139
urge 4457
urgn 7015
uren 4275
ugen 4972
rgen 4959
rget 9844
rgnt 3792
rent 6212
gent 2969
gens 6706
gets 3781
gnts 7414
ents 5231
ent  4134
ens  1555
ets  3782
nts  8968
ntsk 9475
nt k 3220
ns k 4026
ts k 9939
ts i 8120
tski 5940
t ki 9252
s ki 96
s kl 952
s il 3509
skil 6168
 kil 207
 kil 207
 kll 8871
 ill 5967
kill 7850
kile 9756
kile 9756
klle 2496
ille 8489
illd 850
iled 3351
iled 3351
lled 581
lle  4774
lld  9177
led  5938
led  5938
ledi 6987
le i 5468
ld i 923
ed i 9334
ed n 9075
edin 1468
e in 4858
d in 2441
d i  6276
d n  7684
din  7982
 in  9733
 ino 7460
 i o 5078
 n o 1520
in o 7920
in n 5985
inon 3772
i on 3446
n on 4708
n og 4449
n ng 8969
nong 4898
 ong 8293
 ono 8541
 ogo 1270
 ngo 7224
ongo 6272
ongi 753
onoi 556
ogoi 2762
ngoi 9989
ngon 9238
ngin 1365
noin 3719
goin 4391
goig 3637
gong 2883
ging 2883
oing 7458
oin  3720
oig  7941
ong  6973
ing  6512
i

In [22]:
def read_entities_map(datapath, filename):
    uri_map = {}
    f = open(datapath+filename, 'r', encoding="utf8")
    for line in f:
        nr_id, uri = line.split()[0], line.split()[1]
        uri_map[nr_id] = uri
    f.close()
    return uri_map


def get_graph(datapath, triplesfile, ent_map, rel_map):
    graph = {}
    f = open(datapath+triplesfile, 'r')
    for line in f:
        ls = line.split()
        h, r, t = ent_map[ls[0]].split('/')[-1], rel_map[ls[1]].split('/')[-1], ent_map[ls[2]].split('/')[-1]
        graph.setdefault(h, [])
        graph[h].append((r, t))
        graph.setdefault(t, [])
        graph[t].append(('reverse_'+r, h))
    f.close()
    return graph

In [23]:
def load_embeddings(datapath, filename):
    f = open(datapath+filename, 'r', encoding='utf8')
    emb_map = {}
    for line in f:
        ls = line.split()
        vals = [float(val) for val in ls[1:]]
        if len(vals) == 1:
            continue
        emb_map[ls[0]] = vals
    f.close()
    return emb_map

In [24]:
datapath = 'data/fr_en/'

ent_map1 = read_entities_map(datapath, 'ent_ids_1')
ent_map2 = read_entities_map(datapath, 'ent_ids_2')

rel_map1 = read_entities_map(datapath, 'rel_ids_1')
rel_map2 = read_entities_map(datapath, 'rel_ids_2')

G1 = get_graph(datapath, 'triples_1', ent_map1, rel_map1)
G2 = get_graph(datapath, 'triples_2', ent_map2, rel_map2)

datapath_embeddings = 'data/embeddings/'
embs1 = load_embeddings(datapath_embeddings, "deepwalk_embs1.txt")
embs2 = load_embeddings(datapath_embeddings, "deepwalk_embs2.txt")

In [25]:
G1['Rodrigo_Rato'], len(embs1['parti'])

([('prédécesseur', 'Mariano_Rajoy'),
  ('prédécesseur', 'Horst_Köhler'),
  ('successeur', 'María_Teresa_Fernández_de_la_Vega'),
  ('reverse_successeur', 'Pedro_Solbes'),
  ('université', 'Université_complutense_de_Madrid'),
  ('parti', 'Parti_populaire_(Espagne)'),
  ('reverse_prédécesseur', 'José_Montilla'),
  ('prédécesseur', 'Pedro_Solbes'),
  ('fonction', 'Vice-président_du_gouvernement_(Espagne)'),
  ('présidentDuGouvernement', 'José_María_Aznar'),
  ('reverse_successeur', 'Mariano_Rajoy'),
  ('reverse_prédécesseur', 'José_María_Aznar'),
  ('reverse_successeur', 'José_María_Aznar'),
  ('successeur', 'Dominique_Strauss-Kahn'),
  ('reverse_prédécesseur', 'Dominique_Strauss-Kahn')],
 100)

In [26]:
X1 =  np.array(list(embs1.values()))
X2 =  np.array(list(embs2.values()))
X1.shape, X2.shape

((21378, 100), (22176, 100))

In [27]:
idx = 10025
for v1, v2 in zip(embs1[list(embs1.keys())[idx]], X1[idx]):
    assert v1 == v2

In [28]:
hash('ka')%1000

362

In [40]:
def string2vec(string, bins, nr_chars, distance):
    vec = [0 for _ in range(bins)]
    for sg in list(skipgrams(string, nr_chars, distance)):
        vec[hash(''.join(sg))%bins] += 1
    return vec

string2vec('konstantin', 12, 2, 1)

[2, 1, 3, 1, 0, 1, 2, 0, 1, 2, 2, 2]

In [57]:
def get_vectors(G, embs, kernel_map, bins, nr_chars=2, distance=1):
    X =  np.array(list(embs.values()))
    print(X.shape)
    sketch = kernel_map.fit_transform(X)
    assert len(embs) == sketch.shape[0]
    kernel_embs = {entity: kemb for entity, kemb in zip(embs, sketch)}
    cnt = 0
    nf = 0
    entity_kernel_embs = {}
    for node, neighbors in G.items():
        # print(node)
        cnt += 1
        if cnt % 5000 == 0:
            print(cnt)
        emb_head = kernel_embs[node]
        skipgram_head = string2vec(node, bins, nr_chars, distance)
        nbr_rels, nbr_tails = [], []
        skipgram_rels, skipgram_tails = [], []
        for nbr in neighbors:
            if nbr[0] not in embs or nbr[1] not in embs:
                # print('not found', nf, nbr)
                nf += 1
                continue
            emb_rel, emb_tail = kernel_embs[nbr[0]], kernel_embs[nbr[1]]
            nbr_rels.append(emb_rel)
            nbr_tails.append(emb_tail)
            skipgram_rels.append(string2vec(nbr[0], bins, nr_chars, distance))
            skipgram_tails.append(string2vec(nbr[1], bins, nr_chars, distance))
        emb_rel_agg = np.sum(nbr_rels, axis=0)
        emb_tails_agg = np.sum(nbr_tails, axis=0)
        skipgram_rel_agg = np.sum(skipgram_rels, axis=0)
        skipgram_tails_agg = np.sum(skipgram_tails, axis=0)
        v = list(emb_head) + list(emb_rel_agg) + list(emb_tails_agg) + list(skipgram_head) + list(skipgram_rel_agg) + list(skipgram_tails_agg)
        #v = np.array([emb_head, emb_rel_agg, emb_tails_agg, skipgram_head, skipgram_rel_agg, skipgram_tails_agg]).flatten()
        entity_kernel_embs[node] = np.array(v)#/np.linalg.norm(v)
    return entity_kernel_embs


poly_sketch = PolynomialCountSketch(degree=2, n_components=500, random_state=1)

entity_embs1 = get_vectors(G1, embs1, poly_sketch, bins=100)
entity_embs2 = get_vectors(G2, embs2, poly_sketch, bins=100)

(21378, 100)
5000
10000
15000
(22176, 100)
5000
10000
15000


In [58]:
def write_entity_embs_to_file(embeddings_map, path, fname):
    f = open(path + fname, 'w', encoding="utf8")
    for ent, emb in embeddings_map.items():
        f.write(ent + ' ')
        for val in emb:
            f.write(str(val) + ' ')
        f.write('\n')
    f.close()

write_entity_embs_to_file(entity_embs1, 'data/embeddings/', 'final_embs1.txt')
write_entity_embs_to_file(entity_embs2, 'data/embeddings/', 'final_embs2.txt')

In [42]:
entity_embs1.keys()

dict_keys(['Rodrigo_Rato', 'Mariano_Rajoy', 'Robert_Schuman', 'Georges_Bidault', 'Ed_Miliband', 'David_Cameron', 'Call_of_Duty_(série)', 'Wii_U', 'Nothing_but_the_Beat', 'I_Can_Only_Imagine', 'Saint-Évariste-de-Forsyth', 'Courcelles_(Québec)', 'Alexis_Ier_(tsar_de_Russie)', 'Moscou', 'École_normale_supérieure_(Paris)', 'Paris', 'Domingos_Leite_Pereira', 'Porto', 'Big_Thing', 'Duran_Duran', 'Gouvernement_Major', 'Monarchie_britannique', 'Cozy_Powell', 'Rock_instrumental', 'Menino_Jesus', 'Centro_(Santa_Maria)', 'Xenia_Alexandrovna_de_Russie', 'Alexandre_Mikhaïlovitch_de_Russie', 'Europe_Écologie_Les_Verts', 'Parlement_européen', 'Wigan_Athletic_Football_Club', 'Rotherham_United_Football_Club', 'Pierre_III_(empereur_de_Russie)', 'Élisabeth_Ire_(impératrice_de_Russie)', 'Dragon_Age', 'Xbox_360', 'Luke_Ward', 'Josh_Schwartz', 'Borussia_Mönchengladbach', 'Ligue_Europa', 'Izegem', 'Nieuw-Vlaamse_Alliantie', 'Revolution_de_la_Nouvelle-Angleterre', 'Sporting_de_Kansas_City', 'Jiří_Paroubek', '

In [43]:
np.dot(entity_embs1['Elton_John'], entity_embs2['Elton_John'])

815232.0031446901

In [44]:
ent_map1['22711'], ent_map2['34847']

('http://fr.dbpedia.org/resource/Elton_John',
 'http://dbpedia.org/resource/Elton_John')

In [45]:
def read_pairs(path, fname, ent_map1, ent_map2):
    f = open(path+fname, 'r')
    pairs = []
    for line in f:
        ls = line.split('	')
        pairs.append((ent_map1[str(ls[0])].split('/')[-1], ent_map2[str(ls[1]).strip()].split('/')[-1]))
    f.close()
    return pairs

datapath = 'data/fr_en/'
pairs = read_pairs(datapath, 'sup_pairs', ent_map1, ent_map2)

In [46]:
Y = np.array(list(entity_embs2.values()))
Y.shape

(19993, 3300)

In [52]:
nbrs = NearestNeighbors(n_neighbors=100, algorithm='ball_tree').fit(Y)

In [53]:
q = np.array(entity_embs1[pairs[-1][0]]).reshape(1,-1)

In [54]:
distances, indices = nbrs.kneighbors(q)

In [55]:
print(pairs[-1])
enity_names = list(entity_embs2.keys())
for idx in indices[0]:
    print(enity_names[idx])

('Elton_John', 'Elton_John')
Zouk
Deathgrind
Osmose_Productions
Neon_Gold_Records
Kill_Rock_Stars
Studio_One_(record_label)
Konichiwa_Records
Revealed_Recordings
Terror_Squad_Entertainment
Drag_City_(record_label)
Desert_Storm_Records
Sheila_E.
Just_as_Long_as_We're_Together
Spice_Girls
Galway
Kilkenny
Trois-Pistoles,_Quebec
Oleta_Adams
Waterford
Angie_Stone
Ventura,_California
Digne-les-Bains
K.Maro
Beck,_Bogert_&_Appice
Dixie_Chicks
PolyGram_Filmed_Entertainment
Hyde_(musician)
Little_Brother_(group)
Television_(band)
Germs_(band)
Jimmy_Smith_(musician)
Michel_Sardou
Naked_City_(band)
Patrick_Fiori
Steeler_(American_band)
Eric_Prydz
Jerry_Goldsmith
Jaki_Byard
Cat_Power
Syreeta_Wright
Freddie_Gibbs
Girls'_Generation-TTS
Les_Paul
Almah_(band)
Exo_(band)
F(x)_(band)
Geto_Boys
Mano_Negra
Lady_Antebellum
The_Buggles
Radio_Bemba_Sound_System
Mountain_(band)
Alexisonfire
Artie_Shaw
Manfred_Mann's_Earth_Band
Adrar,_Algeria
Montauban
Melon_Kinenbi
Perm_Krai
Saint_Peter
Slint
Sin_City
Second_M

{'10500': 'http://dbpedia.org/resource/Saint-Joseph-de-Coleraine,_Quebec',
 '10501': 'http://dbpedia.org/resource/Self_Portrait_(Bob_Dylan_album)',
 '10502': 'http://dbpedia.org/resource/Alliance_of_Liberals_and_Democrats_for_Europe_Party',
 '10503': 'http://dbpedia.org/resource/Walloon_language',
 '10504': 'http://dbpedia.org/resource/Android_(operating_system)',
 '10505': 'http://dbpedia.org/resource/Part_II_(On_the_Run)',
 '10506': 'http://dbpedia.org/resource/Civic_Choice',
 '10507': 'http://dbpedia.org/resource/Tommy_Jakobsen',
 '10508': 'http://dbpedia.org/resource/Expedition_46',
 '10509': 'http://dbpedia.org/resource/Sugababes',
 '10510': 'http://dbpedia.org/resource/Abel_Goumba',
 '10511': 'http://dbpedia.org/resource/I_Feel_for_You',
 '10512': 'http://dbpedia.org/resource/All_Hope_Is_Gone',
 '10513': 'http://dbpedia.org/resource/Hans_Hoogervorst',
 '10514': 'http://dbpedia.org/resource/List_of_heads_of_state_of_the_Central_African_Republic',
 '10515': 'http://dbpedia.org/reso